# IS 477 Project Molly Babczak

In [64]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score


# Load in Data

In [7]:
import pandas as pd
imdb = pd.read_csv('IMDBTop250Movies.csv')
imdb


,rank,name,year,rating,genre,certificate,run_time,tagline,budget,box_office,casts,directors,writers
0,1,The Shawshank Redemption,1994,9.3,Drama,R,2h 22m,Fear can hold you prisoner. Hope can set you f...,25000000,28884504,"Tim Robbins,Morgan Freeman,Bob Gunton,William ...",Frank Darabont,"Stephen King,Frank Darabont"
1,2,The Godfather,1972,9.2,"Crime,Drama",R,2h 55m,An offer you can't refuse.,6000000,250341816,"Marlon Brando,Al Pacino,James Caan,Diane Keato...",Francis Ford Coppola,"Mario Puzo,Francis Ford Coppola"
2,3,The Dark Knight,2008,9.0,"Action,Crime,Drama",PG-13,2h 32m,Why So Serious?,185000000,1006234167,"Christian Bale,Heath Ledger,Aaron Eckhart,Mich...",Christopher Nolan,"Jonathan Nolan,Christopher Nolan,David S. Goyer"
3,4,The Godfather Part II,1974,9.0,"Crime,Drama",R,3h 22m,All the power on earth can't change destiny.,13000000,47961919,"Al Pacino,Robert De Niro,Robert Duvall,Diane K...",Francis Ford Coppola,"Francis Ford Coppola,Mario Puzo"
4,5,12 Angry Men,1957,9.0,"Crime,Drama",Approved,1h 36m,Life Is In Their Hands -- Death Is On Their Mi...,350000,955,"Henry Fonda,Lee J. Cobb,Martin Balsam,John Fie...",Sidney Lumet,Reginald Rose
...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,246,The Help,2011,8.1,Drama,PG-13,2h 26m,Change begins with a whisper.,25000000,216639112,"Viola Davis,Emma Stone,Octavia Spencer,Bryce D...",Tate Taylor,"Tate Taylor,Kathryn Stockett"
246,247,Dersu Uzala,1975,8.2,"Adventure,Biography,Drama",G,2h 22m,There is man and beast at nature's mercy. Ther...,4000000,14480,"Maksim Munzuk,Yuriy Solomin,Mikhail Bychkov,Vl...",Akira Kurosawa,"Akira Kurosawa,Yuriy Nagibin,Vladimir Arsenev"
247,248,Aladdin,1992,8.0,"Animation,Adventure,Comedy",G,1h 30m,Wish granted! (DVD re-release),Not Available,Not Available,"Scott Weinger,Robin Williams,Linda Larkin,Jona...","Ron Clements,John Musker","Ron Clements,John Musker,Ted Elliott"
248,249,Gandhi,1982,8.0,"Biography,Drama,History",PG,3h 11m,His Triumph Changed The World Forever.,22000000,52767889,"Ben Kingsley,John Gielgud,Rohini Hattangadi,Ro...",Richard Attenborough,John Briley


In [10]:
import os
os.makedirs("../data", exist_ok=True)

In [15]:
import requests
import pandas as pd

API_KEY = "3c4a18b6655f4b82d28873bb68aba431"

url = f"https://api.themoviedb.org/3/movie/top_rated?api_key={API_KEY}&language=en-US&page=1"

response = requests.get(url)
data = response.json()
tmdb = pd.DataFrame(data['results'])
tmdb.to_csv("../data/tmdb_movies.csv", index=False)
tmdb.head()

,adult,backdrop_path,genre_ids,id,original_language,original_title,overview,popularity,poster_path,release_date,title,video,vote_average,vote_count
0,False,/v8xVDqt8uCul3c3mgx4VpGCwxJC.jpg,"[18, 80]",278,en,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,24.7514,/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,1994-09-23,The Shawshank Redemption,False,8.711,29179
1,False,/jdHsptJbtalEuVhCV5i7kSC3g0x.jpg,"[18, 80]",238,en,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",25.3535,/3bhkrj58Vtu7enYsRolD1fZdja1.jpg,1972-03-14,The Godfather,False,8.685,22041
2,False,/kGzFbGhp99zva6oZODW5atUtnqi.jpg,"[18, 80]",240,en,The Godfather Part II,In the continuing saga of the Corleone crime f...,13.4993,/hek3koDUyRQk7FIhPXsa6mT2Zc3.jpg,1974-12-20,The Godfather Part II,False,8.571,13322
3,False,/zb6fM1CX41D9rF9hdgclu0peUmy.jpg,"[18, 36, 10752]",424,en,Schindler's List,The true story of how businessman Oskar Schind...,10.5948,/sF1U4EUQS8YHUYjNl3pMGNIQyr0.jpg,1993-12-15,Schindler's List,False,8.566,16845
4,False,/tj6iPnz18hGfr0LKqWmG6Cp3niO.jpg,[18],389,en,12 Angry Men,The defense and the prosecution have rested an...,8.9603,/ow3wq89wM8qd5X7hWKxiRfsFf9C.jpg,1957-04-10,12 Angry Men,False,8.549,9513


# Explore Data

In [16]:
print("IMDb shape:", imdb.shape)
print("TMDb shape:", tmdb.shape)

IMDb shape: (250, 13)
TMDb shape: (20, 14)


In [18]:
print("IMDb columns:", imdb.columns)
print("TMDb columns:", tmdb.columns)

IMDb columns: Index(['rank', 'name', 'year', 'rating', 'genre', 'certificate', 'run_time',
       'tagline', 'budget', 'box_office', 'casts', 'directors', 'writers'],
      dtype='object')
TMDb columns: Index(['adult', 'backdrop_path', 'genre_ids', 'id', 'original_language',
       'original_title', 'overview', 'popularity', 'poster_path',
       'release_date', 'title', 'video', 'vote_average', 'vote_count'],
      dtype='object')


# Combine Data

In [55]:
imdb_subset = imdb[['name', 'year', 'genre', 'rating', 'budget', 'box_office']].copy()
tmdb_subset = tmdb[['title', 'release_date', 'genre_ids', 'vote_average', 'vote_count', 'popularity']].copy()

imdb_subset.columns = imdb_subset.columns.str.lower()
tmdb_subset.columns = tmdb_subset.columns.str.lower()

# rename
tmdb_subset = tmdb_subset.rename(columns={
    'title': 'name_tmdb',
    'release_date': 'release_date_tmdb',
    'vote_average': 'vote_average_tmdb',
    'vote_count': 'vote_count_tmdb'
})

#  target column
imdb_subset['target_rating'] = imdb_subset['rating']
tmdb_subset['target_rating'] = tmdb_subset['vote_average_tmdb']

imdb_subset['genre_ids'] = None
imdb_subset['popularity'] = None
imdb_subset['vote_count_tmdb'] = None
imdb_subset['release_date_tmdb'] = None

tmdb_subset['rating'] = None
tmdb_subset['budget'] = None
tmdb_subset['box_office'] = None
tmdb_subset['genre'] = None

combined = pd.concat([imdb_subset, tmdb_subset], ignore_index=True, sort=False)
print("Combined shape:", combined.shape)


Combined shape: (270, 13)


/var/folders/0n/dfqjq4691b3f0zm3nqj0j6g00000gn/T/ipykernel_14377/3962273084.py:29: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([imdb_subset, tmdb_subset], ignore_index=True, sort=False)


# Data Cleaning and Preparation for Machine Learning 

In [56]:
combined['name'].unique

<bound method Series.unique of 0      the shawshank redemption
1                 the godfather
2               the dark knight
3         the godfather part ii
4                  12 angry men
                 ...           
265                         NaN
266                         NaN
267                         NaN
268                         NaN
269                         NaN
Name: name, Length: 270, dtype: object>

In [74]:
numeric_features = ['budget', 'box_office', 'tmdb_votes', 'release_year']
categorical_features = ['Action', 'Drama', 'Comedy', 'Crime', 'Adventure', 'Biography', 'Fantasy', 'Animation', 'Horror', 'Romance'] 

imdb['target_rating'] = imdb['rating']
tmdb['target_rating'] = tmdb['vote_average']


numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])


In [69]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [70]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('regressor', model)])


combined['release_year'] = combined.apply(
    lambda row: row['year'] if pd.notnull(row['year']) else (pd.to_datetime(row['release_date_tmdb']).year if pd.notnull(row['release_date_tmdb']) else None),
    axis=1
)
top_genres = ['Action', 'Drama', 'Comedy', 'Crime', 'Adventure', 'Biography', 'Fantasy', 'Animation', 'Horror', 'Romance']
for genre in top_genres:
    combined[genre] = combined['genre'].apply(lambda x: 1 if x is not None and genre in x else 0)

combined['budget'] = pd.to_numeric(combined['budget'], errors='coerce')
combined['box_office'] = pd.to_numeric(combined['box_office'], errors='coerce')
combined['vote_count_tmdb'] = pd.to_numeric(combined['vote_count_tmdb'], errors='coerce')
combined['popularity'] = pd.to_numeric(combined['popularity'], errors='coerce')

In [71]:
numeric_features = ['budget', 'box_office', 'vote_count_tmdb', 'popularity', 'release_year']
categorical_features = top_genres
target = 'target_rating'

X = combined[numeric_features + categorical_features]
y = combined[target]

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

model = RandomForestRegressor(n_estimators=100, random_state=42)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', model)
])

# Train/Test Split

In [72]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['budget', 'box_office',
                                                   'vote_count_tmdb',
                                                   'popularity',
                                                   'release_year']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value=0,
                                                                                 strategy='constant'))]),
                                                  ['Action', 'Drama', 'Comedy',
                                                   'Crime', 'Adventure',
                                                   'Biography', 'Fantasy',
                                                   'Animation', 'Horror',
                                                   'Romance'])])),
                ('regressor', RandomForestRegressor(random_state=42))])

# Make Predictions 

In [73]:
preds = pipeline.predict(X_test)
r2 = r2_score(y_test, preds)
print(f"R² score on test set: {r2:.3f}")

R² score on test set: 0.148
